# Лабораторная работа 1. Инструменты, данные и первый ориентир

**Курс «Машинное обучение», 4 курс**

| | |
|---|---|
| Место в курсе | первое занятие курса, **до** лекции 1 |
| Опора на лекции | теории курса не требуется; нужны линейная алгебра и базовый Python. Приёмы NumPy (ось, срез, broadcasting, матричное умножение) напоминаются по ходу — см. врезку перед первым заданием |
| Трудоёмкость | 2 ч аудиторно + домашняя работа (3–4 ч) |

## Цель работы

Пройти путь от сырой таблицы до матрицы «объекты–признаки», пригодной для обучения, и получить первый ориентир качества, с которым дальше сравниваются все модели курса. Попутно освоить векторизацию в NumPy — приём, без которого не обойдётся ни одно занятие.

## Как устроено занятие

Ноутбук разбирается в аудитории: код запускается и обсуждается по ходу.
Большая часть ячеек уже написана — их нужно **прочитать и запустить**,
разобравшись, что происходит и почему.

Ячейки, помеченные `# ✍ ЗАДАНИЕ НА СЕМИНАРЕ`, заполняются самостоятельно
прямо на занятии; их немного, и каждая занимает несколько строк. Ячейки
**Вывод** — тоже ваши: короткий ответ своими словами на поставленный вопрос.

Дома выполняется отдельный ноутбук `lab01_homework.ipynb` — там
задания крупнее и делать их нужно самому.

> **Данные занятия — учебные, одинаковые у всех.** Числа на экране у преподавателя
> и у вас совпадают, поэтому можно сверяться с соседом и спорить о результате вслух.
> Индивидуальная таблица, порождённая по вашему ФИО, появится в домашней работе.

## Как устроен практикум

Девять занятий, чередующихся с лекциями, причём курс начинается практикумом:

```
Зан. 1 → Лек 1 → Зан. 2 → Лек 2 → … → Лек 8 → Зан. 9
```

Значит, занятие $k+1$ закрепляет лекцию $k$. Сегодняшнее идёт до первой лекции
и теории курса не требует — это подготовка инструментов и данных.

**Данные на занятии и дома разные.** В аудитории мы работаем с учебной
таблицей, одинаковой у всех: так можно сверить число с соседом и обсудить
результат вслух. Дома у каждого своя таблица, порождённая по ФИО, — приёмы те
же, числа свои.

In [ ]:
# Служебная ячейка: импорты, стиль графиков, воспроизводимость.
import sys, pathlib, warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)

# Модули практикума (variants.py, labdata.py) ищем рядом с ноутбуком.
# Если их нет -- значит, ноутбук открыт в Colab: скачиваем из репозитория курса.
COURSE_FILES_URL = "https://raw.githubusercontent.com/sharipovaka/mltest1/main/notebooks"


def _course_modules_dir():
    here = pathlib.Path.cwd()
    for parent in [here, *here.parents][:4]:
        if (parent / "variants.py").exists():
            return str(parent)
    import urllib.request
    for name in ("variants.py", "labdata.py"):
        if not pathlib.Path(name).exists():
            urllib.request.urlretrieve(f"{COURSE_FILES_URL}/{name}", name)
            print(f"загружен {name} из репозитория курса")
    return str(here)


sys.path.insert(0, _course_modules_dir())
from variants import get_variant, describe_variant  # noqa: E402

RANDOM_STATE = 42          # единый seed на всю работу: результаты воспроизводимы
rng = np.random.default_rng(RANDOM_STATE)

plt.rcParams.update({
    "figure.figsize": (7.5, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("numpy", np.__version__, "| pandas", pd.__version__)

from labdata import load_titanic

---
# Часть 1. Векторизация вместо циклов

Курс записан в матричных обозначениях: матрица «объекты–признаки» размера
$\ell\times n$ — это массив NumPy формы `(l, n)`. Правило, которое сэкономит
десятки часов: **если вы пишете цикл по объектам выборки, почти наверняка есть
векторизованная запись — короче и в десятки раз быстрее.**

Проверим это на матрице попарных квадратов расстояний
$D_{ij} = \|a_i - b_j\|^2$. Двойной цикл здесь не нужен, потому что

$$
\|a - b\|^2 = \|a\|^2 - 2\langle a, b\rangle + \|b\|^2 .
$$

> **Напоминание — NumPy: ось, срез, broadcasting и `@`.** Четыре приёма, без которых дальше не обойтись. Пусть `X` имеет форму `(l, n)`.
>
> **Ось.** `X.sum(axis=0)` даёт $n$ чисел — по одному на признак, `X.sum(axis=1)`
> даёт $\ell$ чисел — по одному на объект. Правило: `axis` указывает, какая ось
> **исчезает**.
>
> **Срез.** `X[i]` — $i$-й объект (строка), `X[:, j]` — $j$-й признак (столбец).
>
> **Broadcasting.** Ось длины 1 растягивается: `(l, 1) + (1, k)` даёт `(l, k)`.
> Отсюда приём `[:, None]` — превращает вектор в столбец, `[None, :]` — в строку.
> Столбец плюс строка есть таблица всех попарных сумм, и всё это без цикла.
>
> **Матричное умножение.** `A @ B` — произведение матриц, `A * B` — поэлементное.
> Путать их — самая частая ошибка первых недель.
>
> Если что-то не сходится, первым делом печатайте `.shape`.

In [ ]:
import time


def dists_loops(A, B):
    """Попарные квадраты расстояний двойным циклом -- эталон корректности."""
    D = np.empty((A.shape[0], B.shape[0]))
    for i in range(A.shape[0]):
        for j in range(B.shape[0]):
            diff = A[i] - B[j]
            D[i, j] = diff @ diff
    return D

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def dists_vectorized(A, B):
    """То же самое без единого цикла."""
    a2 = np.sum(A ** 2, axis=1)[:, None]     # столбец (m, 1): квадраты норм строк A
    b2 = np.sum(B ** 2, axis=1)[None, :]     # строка  (1, k): квадраты норм строк B
    # TODO (2 строки): D = a2 - 2 * (скалярные произведения всех пар) + b2,
    #                  скалярные произведения -- это A @ B.T;
    #                  верните np.maximum(D, 0.0) -- почему, см. вывод ниже.
    raise NotImplementedError

In [ ]:
A = rng.normal(size=(400, 20))
B = rng.normal(size=(300, 20))

t0 = time.perf_counter(); D_loop = dists_loops(A, B); t_loop = time.perf_counter() - t0
t0 = time.perf_counter(); D_vec = dists_vectorized(A, B); t_vec = time.perf_counter() - t0

print(f"максимальное расхождение: {np.abs(D_loop - D_vec).max():.2e}")
print(f"цикл:         {t_loop * 1000:7.0f} мс")
print(f"векторизация: {t_vec * 1000:7.0f} мс  ->  быстрее в {t_loop / t_vec:.0f} раз")

> **Вывод.** Во сколько раз быстрее? Почему расхождение не строго нулевое и зачем `np.maximum(D, 0)`?
>
> *(ваш ответ здесь)*

---
# Часть 2. Что лежит в таблице

Учебная таблица занятия — список пассажиров «Титаника» (891 строка).
Задача: предсказать `survived` — выжил пассажир или нет.

Датасет выбран не за романтику, а за честную грязь: в нём есть пропуски,
повторяющиеся строки, признаки-дубликаты и один столбец, который вообще нельзя
подавать в модель. Ровно это встречается в любой реальной выгрузке.

In [ ]:
df_raw = load_titanic()
TARGET = "survived"

print(f"объектов: {df_raw.shape[0]}, столбцов: {df_raw.shape[1]}")
print(f"полных дубликатов строк: {df_raw.duplicated().sum()}")
df_raw.head()

In [ ]:
# Сводка по столбцам: тип, сколько разных значений, сколько пропусков
info = pd.DataFrame({
    "тип": df_raw.dtypes.astype(str),
    "уникальных": df_raw.nunique(),
    "пропусков, %": (df_raw.isna().mean() * 100).round(1),
})
display(info)

> **Вывод.** Почему `deck` в таком виде почти бесполезна? И что делать со 124 повторяющимися строками — это ошибка выгрузки или разные пассажиры?
>
> *(ваш ответ здесь)*

---
# Часть 3. Смотрим на данные глазами

Таблицу мы прочитали, но ещё не видели. Разведочный анализ отвечает на три
вопроса, и на каждый есть свой график:

| Вопрос | График |
|---|---|
| как устроен отдельный признак | гистограмма |
| различает ли признак классы | ящик с усами по классам |
| как признаки связаны между собой | тепловая карта корреляций |

Третий вопрос сегодня главный: именно карта корреляций найдёт нам то, что мы
пока не заметили.

In [ ]:
# Вопрос 1: как устроен отдельный признак
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

for ax, col in zip(axes, ("age", "fare")):
    ax.hist(df_raw[col].dropna(), bins=30)
    ax.set_xlabel(col)
    ax.set_ylabel("объектов")
    ax.set_title(f"{col}: скошенность {df_raw[col].skew():+.2f}")

plt.tight_layout(); plt.show()

In [ ]:
# Вопрос 2: различает ли признак классы
by_class = [df_raw.loc[df_raw[TARGET] == k, "fare"].dropna() for k in (0, 1)]

fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.boxplot(by_class, tick_labels=["погиб", "выжил"])
ax.set_yscale("log")               # тариф сильно скошен: на линейной оси не видно
ax.set_ylabel("fare")
ax.set_title("Тариф по исходу")
plt.tight_layout(); plt.show()

display(df_raw.groupby(TARGET)[["age", "fare"]].median().round(1))

> **Вывод.** Какой из двух признаков различает выживших, а какой почти нет? Зачем на тарифе логарифмическая шкала?
>
> *(ваш ответ здесь)*

### Задание 3.1. Найти признаки, дублирующие друг друга

Корреляция Пирсона считается только для чисел, поэтому сначала закодируем
бинарные и порядковые признаки, а потом посмотрим на все пары сразу.
Нас интересуют пары с $|r|$ близким к единице — это признаки, несущие **одну и
ту же** информацию.

In [ ]:
d = df_raw.copy()
d["alive"] = d["alive"].map({"yes": 1, "no": 0})
d["class"] = d["class"].map({"First": 1, "Second": 2, "Third": 3})
d["alone"] = d["alone"].astype(int)

C = d.select_dtypes(include="number").corr()      # матрица корреляций
print("признаков в карте:", len(C))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(C)), C.columns, rotation=90)
ax.set_yticks(range(len(C)), C.columns)
fig.colorbar(im, label="коэффициент корреляции")
ax.set_title("Связи между признаками")
plt.tight_layout(); plt.show()

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

pairs = []
for i, a in enumerate(C.columns):
    for b in C.columns[i + 1:]:           # каждую пару берём ровно один раз
        # TODO (1 строка): добавьте в pairs словарь
        #   {"признак A": a, "признак B": b, "|r|": abs(C.loc[a, b])}
        pass

# TODO (2 строки): соберите из pairs таблицу, отсортируйте по |r| по убыванию
#                  и покажите первые пять строк

> **Вывод.** Две пары дали $|r| = 1.000$. Что это за признаки и что с ними делать? Чем от них отличается пара `fare`–`class` с $r = -0.55$?
>
> *(ваш ответ здесь)*

---
# Часть 4. Признак, которого не должно быть

Вернёмся к находке: `alive` совпадает с целевой переменной идеально. Формально
это обычный категориальный признак со значениями `yes`/`no`. Посмотрим, что
будет, если подать в модель только его.

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier

alive01 = df_raw["alive"].map({"yes": 1, "no": 0}).to_frame()
tree = DecisionTreeClassifier(max_depth=3, random_state=RANDOM_STATE)
score = cross_val_score(tree, alive01, df_raw[TARGET], cv=5, scoring="roc_auc")

print(f"ROC-AUC по ОДНОМУ признаку alive: {score.mean():.4f}")
print("alive совпадает с survived на всех объектах:",
      (alive01["alive"] == df_raw[TARGET]).all())

In [ ]:
# Убираем утечку и два признака-дубликата: из каждой пары оставляем один
df = df_raw.drop(columns=["alive",          # копия целевой переменной
                          "class",          # то же, что pclass, только словами
                          "embark_town"])   # то же, что embarked, только названием
df["alone"] = df["alone"].astype(int)       # бинарный признак кодируем 0/1

print("осталось признаков:", df.shape[1] - 1)
print(list(df.columns.drop(TARGET)))

> **Вывод.** Откуда идеальное качество и почему `alive` необходимо удалить, хотя он «лучший признак в таблице»?
>
> *(ваш ответ здесь)*

---
# Часть 5. Типы признаков и кодирование

Тип признака определяет, что с ним вообще можно делать:

| Тип | Множество значений | Пример в таблице | Что осмысленно |
|---|---|---|---|
| бинарный | $\{0,1\}$ | `alone` | всё |
| номинальный | конечное, без порядка | `embarked`, `deck` | сравнение на равенство |
| порядковый | конечное, упорядоченное | `pclass` | сравнение $\le$ |
| количественный | $\mathbb{R}$ | `age`, `fare` | арифметика |

Ошибка здесь стоит дорого и в обе стороны:

* закодировав **номинальный** признак числами 1, 2, 3, мы сообщаем модели
  порядок, которого нет: Шербур окажется «между» Саутгемптоном и Квинстауном;
* закодировав **порядковый** признак индикаторами (one-hot), мы, наоборот,
  выбрасываем информацию: модель больше не знает, что второй класс между
  первым и третьим.

`pclass` — порядковый (1 лучше 2 лучше 3), `embarked` — номинальный.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

# Номинальный -> индикаторы; порядковый оставляем числом, порядок задаём мы
E = OneHotEncoder(sparse_output=False).fit_transform(df[["embarked"]].fillna("?"))

print("embarked ->", E.shape[1], "столбцов-индикаторов")
print("сумма индикаторов по строке всегда:", np.unique(E.sum(axis=1)))
print("pclass:", sorted(df["pclass"].unique().tolist()), "-- порядок уже закодирован")

> **Вывод.** Почему для `embarked` нельзя обойтись одним столбцом с числами 1, 2, 3? И почему сумма индикаторов, равная единице, — это повод для беспокойства?
>
> *(ваш ответ здесь)*

---
# Часть 6. Пропуски, выбросы и масштаб

Три вещи, которые надо сделать с числовыми признаками, — и в каждой есть
ловушка.

> **Напоминание — квантиль, квартиль, IQR.** Квантиль уровня $q$ — значение $x_q$, ниже которого лежит доля $q$ наблюдений:
> $x_{0.5}$ — медиана, $x_{0.99}$ — 99-й перцентиль. Квартили — квантили уровней
> $0.25$, $0.5$, $0.75$, обозначаются $Q_1, Q_2, Q_3$. Межквартильный размах
> $\mathrm{IQR} = Q_3 - Q_1$ — ширина «средней половины» данных. Он устойчив к
> выбросам: пока экстремальных значений меньше четверти, границы $Q_1$ и $Q_3$ не
> сдвинутся, тогда как $\overline x$ и $\sigma$ один выброс тянет за собой.

In [ ]:
# (а) Пропуски: сколько стоит "просто удалить строки"
variants_na = {
    "как есть": df,
    "dropna() по всем столбцам": df.dropna(),
    "сначала убрать deck, потом dropna()": df.drop(columns=["deck"]).dropna(),
}

for name, d in variants_na.items():
    print(f"{name:36s} строк {len(d):4d}, доля выживших {d[TARGET].mean():.3f}")

> **Вывод.** Что произошло с долей выживших при `dropna()` по всем столбцам и почему? Как это связано с 77 % пропусков в `deck`?
>
> *(ваш ответ здесь)*

In [ ]:
# (б) Выбросы: два правила дают очень разный ответ
for col in ("age", "fare"):
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    n_iqr = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    n_sigma = (np.abs(s - s.mean()) > 3 * s.std()).sum()
    print(f"{col:5s}: 1.5 IQR -> {n_iqr:3d} объектов, "
          f"3 sigma -> {n_sigma:3d}, скошенность {s.skew():+.2f}")

> **Вывод.** Почему на `fare` правила расходятся в шесть раз? Надо ли удалять эти 116 объектов?
>
> *(ваш ответ здесь)*

### Задание 6.1. Своя стандартизация

$z = (x - \mu)/\sigma$ — приведение признака к нулевому среднему и единичной
дисперсии. Нужно оно не для красоты: на занятии 3 мы увидим, что от масштаба
признаков напрямую зависит, сойдётся ли градиентный спуск.

Реализуйте сами и сверьте со `StandardScaler`. **Это первая из девяти таких
сверок в курсе**: своя реализация обязана совпадать с библиотечной численно,
а не «примерно».

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

def standardize(X, mu=None, sigma=None):
    """z = (x - mu) / sigma. Если mu и sigma не заданы -- оценить их по X."""
    X = np.asarray(X, dtype=float)
    # TODO (3 строки):
    #   1) если mu is None -- посчитать mu = X.mean(axis=0) и sigma = X.std(axis=0);
    #   2) нулевые sigma заменить на 1: np.where(sigma > 0, sigma, 1.0);
    #   3) вернуть (X - mu) / sigma, mu, sigma
    raise NotImplementedError

In [ ]:
from sklearn.preprocessing import StandardScaler

X_num = df[["age", "fare"]].fillna(df[["age", "fare"]].median()).to_numpy()
Z_my, mu, sigma = standardize(X_num)
Z_sk = StandardScaler().fit_transform(X_num)

print(f"max|своя - sklearn| = {np.abs(Z_my - Z_sk).max():.2e}")
print("совпадает численно:", np.allclose(Z_my, Z_sk))

> **Вывод.** Зачем `standardize` принимает $\mu$ и $\sigma$ параметрами, а не считает их всегда заново?
>
> *(ваш ответ здесь)*

---
# Часть 7. Честное разбиение и первый ориентир

Главное правило всего курса:

> Все параметры предобработки — средние, дисперсии, медианы для заполнения
> пропусков, список категорий — вычисляются **только по обучающей части**
> и затем применяются к контрольной.

Иначе информация о контрольной выборке просачивается в обучение, и оценка
качества оказывается завышенной. Насколько именно — измерим на занятии 5.
Порядок действий: **сначала делим, потом обрабатываем**, а не наоборот.

> **Напоминание — стратификация.** При случайном делении выборки доли классов в частях могут заметно разойтись —
> особенно если один класс редкий. Стратифицированное разбиение делит выборку
> внутри каждого класса отдельно, поэтому доли сохраняются. В `sklearn` это
> аргумент `stratify=y`. Для регрессии он не нужен (классов нет), а для
> классификации — практически всегда.

In [ ]:
from sklearn.model_selection import train_test_split

X, y = df.drop(columns=[TARGET]), df[TARGET]
num_cols = list(X.select_dtypes(include="number").columns)
cat_cols = [c for c in X.columns if c not in num_cols]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y)

print(f"обучающая {len(X_train)}, контрольная {len(X_test)}")
print("числовые:", num_cols)
print("категориальные:", cat_cols)

### Задание 7.1. Сборка предобработки

Числовые и категориальные столбцы обрабатываются по-разному, поэтому нужны две
ветки: своя цепочка действий для каждой группы столбцов. `Pipeline` — цепочка,
`ColumnTransformer` — распределитель столбцов по веткам. Оба запоминают
параметры на `fit` и применяют их в `transform`, поэтому правило «параметры
только по обучающей части» выполняется само.

Ветка для чисел уже собрана — соберите по образцу ветку для категорий.

In [ ]:
# ✍ ЗАДАНИЕ НА СЕМИНАРЕ — допишите эту ячейку

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Ветка для чисел: заполнить пропуски медианой, затем стандартизовать
num_branch = Pipeline([("impute", SimpleImputer(strategy="median")),
                       ("scale", StandardScaler())])

# TODO (2 строки): по образцу соберите ветку для категорий --
#   SimpleImputer(strategy="most_frequent"), затем
#   OneHotEncoder(handle_unknown="ignore", sparse_output=False)
cat_branch = ...

preprocessor = ColumnTransformer([("num", num_branch, num_cols),
                                  ("cat", cat_branch, cat_cols)])

In [ ]:
X_train_t = preprocessor.fit_transform(X_train)   # fit ТОЛЬКО на обучающей части
X_test_t = preprocessor.transform(X_test)         # на контрольной -- только transform
feature_names = preprocessor.get_feature_names_out()

print(f"матрица объектов-признаков: {X_train_t.shape} (было {X_train.shape[1]} столбцов)")
print("первые признаки:", list(feature_names[:6]))

### Ориентир: с чем сравнивать модели

Матрица готова — можно обучать. Но прежде чем радоваться качеству, надо знать,
сколько даёт **тривиальный ответ**: для классификации это самый частый класс,
для регрессии — среднее.

Строгие определения функции потерь и риска будут на лекции 1; пока пользуемся
привычными: доля правильных ответов и ROC-AUC.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

models = {"константа (самый частый класс)": DummyClassifier(strategy="most_frequent"),
          "логистическая регрессия": LogisticRegression(max_iter=2000)}

results = {}
for name, m in models.items():
    m.fit(X_train_t, y_train)
    results[name] = {"accuracy": accuracy_score(y_test, m.predict(X_test_t)),
                     "ROC-AUC": roc_auc_score(y_test, m.predict_proba(X_test_t)[:, 1])}
display(pd.DataFrame(results).T.round(4))

> **Вывод.** Насколько модель лучше константы? Чему равен ROC-AUC у константы и почему именно столько?
>
> *(ваш ответ здесь)*

In [ ]:
# Каталог output/ -- то, что ноутбук создаёт сам (курсовые данные лежат в data/)
import pathlib, joblib

out_dir = pathlib.Path("output"); out_dir.mkdir(exist_ok=True)
np.savez_compressed(out_dir / "titanic_prepared.npz", X_train=X_train_t,
                    X_test=X_test_t, y_train=np.asarray(y_train),
                    y_test=np.asarray(y_test),
                    feature_names=np.asarray(feature_names, dtype=object))
joblib.dump(preprocessor, out_dir / "titanic_preprocessor.joblib")

### Мостик к лекции 1

Следующее занятие начнётся с МНК на выборке из восьми кошек ($f_1$ — банок
корма в день, $f_2$ — возраст, $y$ — вес). Эту задачу ставит пример 1.4
конспекта, но чисел там нет — их задаём мы. Наберём таблицу прямо сейчас: на
занятии 2 первым делом надо будет получить $\theta^*$ двумя способами и
убедиться, что они совпадают. Так получается **эталон** — задача с заранее
известным ответом, на которой проверяют реализацию.

Обратите внимание на размер: восемь объектов и три параметра. Эталон не обязан
быть большим — он обязан быть **проверяемым**.

In [ ]:
cats = pd.DataFrame({
    "банок_в_день": [1, 2, 2, 3, 2, 3, 1, 4],
    "возраст": [1, 2, 3, 5, 7, 10, 2, 8],
    "вес": [3.0, 4.2, 4.5, 5.5, 4.8, 6.0, 3.4, 6.6],
})
cats.to_csv(out_dir / "cats.csv", index=False)
print("сохранено в output/:", *[f.name for f in sorted(out_dir.iterdir())])
display(cats)

## Итоги занятия

Ответьте письменно на контрольные вопросы (по 2–4 предложения):

1. Чем номинальный признак отличается от порядкового и почему их нельзя кодировать одинаково? Приведите по примеру из таблицы «Титаника».
2. Что такое утечка данных? Приведите два примера: `alive` из сегодняшней таблицы и придуманный вами — для задачи «вернёт ли клиент кредит».
3. Почему параметры масштабирования оценивают только по обучающей выборке? Что именно завышается, если это правило нарушить?
4. Вы получили accuracy 0.93. Какие ещё два числа нужно знать, чтобы понять, хороший это результат?
5. На карте корреляций `alone` и `sibsp` дают $|r| = 0.58$ — далеко от единицы. При этом `alone` в точности равен `sibsp + parch == 0`. Почему малое $|r|$ не доказывает, что признаки несут разную информацию?

---

**Дома:** откройте `lab01_homework.ipynb` — там три задачи на вашей собственной таблице: разведка и поиск утечки, заполнение пропусков и своя реализация One-Hot.